In [1]:
import pandas as pd
import h5py

In [ ]:
# hdf5_data_path = "data/merge.hdf5"

# with h5py.File(hdf5_data_path, 'r') as f:
#     # List all root-level keys (groups and datasets)
#     print("Keys:", list(f.keys()))
#     group = f["data"]
#     print("Inside 'data':", list(group.keys()))

In [4]:
import h5py
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Absolute paths for your specific user directory on Expanse
hdf5_path = "/expanse/lustre/projects/uci157/ysuh2/data/merge.hdf5"
csv_path = "/expanse/lustre/projects/uci157/ysuh2/data/merge.csv"
output_parquet = "/expanse/lustre/projects/uci157/ysuh2/data/stead_combined.parquet"

# 1. Load CSV and set the index
print("Loading full CSV metadata...")
metadata_df = pd.read_csv(csv_path, low_memory=False)
metadata_df = metadata_df.dropna(subset=['trace_name'])
metadata_df.set_index('trace_name', inplace=True) 

# ==========================================
# THE FIX: Create a Strict Master Schema
# ==========================================
print("Generating master PyArrow schema...")
# Convert the full metadata DF to Arrow to capture the perfect, global data types
meta_table = pa.Table.from_pandas(metadata_df.reset_index())
schema_fields = list(meta_table.schema)

# Add our custom waveform array column to the schema
schema_fields.append(pa.field('waveform_data', pa.list_(pa.float64())))

# Build the final strict schema
master_schema = pa.schema(schema_fields)
# ==========================================

writer = None
chunk_size = 2000

print("Opening HDF5 file...")
with h5py.File(hdf5_path, 'r') as f:
    data_group = f['data']
    
    print("Extracting keys in disk-order (this may take a minute)...")
    hdf5_keys = list(data_group.keys())
    print(f"Found {len(hdf5_keys)} traces in HDF5. Starting extraction...")
    
    for i in range(0, len(hdf5_keys), chunk_size):
        chunk_keys = hdf5_keys[i:i + chunk_size]
        
        waveforms = []
        valid_keys = []
        
        for key in chunk_keys:
            if key in metadata_df.index: 
                ds = data_group[key]
                waveforms.append(np.array(ds).flatten().tolist())
                valid_keys.append(key)
        
        if not valid_keys:
            continue
            
        chunk_df = metadata_df.loc[valid_keys].copy()
        chunk_df.reset_index(inplace=True) 
        chunk_df['waveform_data'] = waveforms
        
        # ==========================================
        # THE FIX: Apply the Master Schema
        # ==========================================
        # This forces the chunk to obey the global types, preventing "null" vs "string" crashes
        chunk_table = pa.Table.from_pandas(chunk_df, schema=master_schema)
        
        if writer is None:
            writer = pq.ParquetWriter(output_parquet, master_schema)
            
        writer.write_table(chunk_table)
        print(f"Processed {min(i + chunk_size, len(hdf5_keys))} / {len(hdf5_keys)} HDF5 arrays...")

if writer:
    writer.close()
    
print("Finished! Master dataset saved to:", output_parquet)

Loading full CSV metadata...
Generating master PyArrow schema...
Opening HDF5 file...
Extracting keys in disk-order (this may take a minute)...
Found 1265657 traces in HDF5. Starting extraction...
Processed 2000 / 1265657 HDF5 arrays...
Processed 4000 / 1265657 HDF5 arrays...
Processed 6000 / 1265657 HDF5 arrays...
Processed 8000 / 1265657 HDF5 arrays...
Processed 10000 / 1265657 HDF5 arrays...
Processed 12000 / 1265657 HDF5 arrays...
Processed 14000 / 1265657 HDF5 arrays...
Processed 16000 / 1265657 HDF5 arrays...
Processed 18000 / 1265657 HDF5 arrays...
Processed 20000 / 1265657 HDF5 arrays...
Processed 22000 / 1265657 HDF5 arrays...
Processed 24000 / 1265657 HDF5 arrays...
Processed 26000 / 1265657 HDF5 arrays...
Processed 28000 / 1265657 HDF5 arrays...
Processed 30000 / 1265657 HDF5 arrays...
Processed 32000 / 1265657 HDF5 arrays...
Processed 34000 / 1265657 HDF5 arrays...
Processed 36000 / 1265657 HDF5 arrays...
Processed 38000 / 1265657 HDF5 arrays...
Processed 40000 / 1265657 HD

In [3]:
metadata_df.head(1)

,network_code,receiver_code,receiver_type,receiver_latitude,receiver_longitude,receiver_elevation_m,p_arrival_sample,p_status,p_weight,p_travel_sec,...,source_magnitude_type,source_magnitude_author,source_mechanism_strike_dip_rake,source_distance_deg,source_distance_km,back_azimuth_deg,snr_db,coda_end_sample,trace_start_time,trace_category
trace_name,,,,,,,,,,,,,,,,,,,,,
109C.TA_201510210555_NO,TA,109C,HH,32.8889,-117.1051,150.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-10-21 05:55:00,noise


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import size, col

# 1. Initialize your Expanse Spark Cluster
spark = SparkSession.builder \
    .appName("STEAD_Parquet_EDA") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "20g") \
    .config("spark.executor.instances", 6) \
    .getOrCreate()


In [4]:
(128-4)/6

20.666666666666668

In [2]:
import time

In [3]:
st = time.time()
# 2. Load the Parquet file
# parquet_path = "/expanse/lustre/projects/uci157/ysuh2/data/stead_combined.parquet"
parquet_path = "/scratch/ysuh2/job_48580420/stead_combined.parquet"
print(f"Loading Parquet from: {parquet_path}")
df = spark.read.parquet(parquet_path)

# 3. Basic Verifications
print("\n--- Total Row Count ---")
# This should print ~1.26 million very quickly!
print(f"Total records: {df.count()}") 

print("\n--- Schema Check ---")
# This will show you all the columns, including your 'waveform_data' array
df.printSchema()

print("\n--- First 5 Rows (Selected Columns) ---")
# Let's peek at the trace name, magnitude, and the actual waveform array
df.select("trace_name", "source_magnitude", "waveform_data").show(5)

# 4. Advanced Verification: Checking the Array Size
print("\n--- Verifying Array Length ---")
# This ensures your 6000x3 matrix was correctly flattened into 18,000 items
df.select("trace_name", size("waveform_data").alias("array_length")).show(5)

# 5. Fun Query: Find the biggest earthquakes!
print("\n--- Top 5 Largest Earthquakes ---")
df.filter(col("source_magnitude").isNotNull()) \
  .orderBy(col("source_magnitude").desc()) \
  .select("trace_name", "source_magnitude", "source_depth_km") \
  .show(5)

print(time.time()-st)

Loading Parquet from: /scratch/ysuh2/job_48580420/stead_combined.parquet

--- Total Row Count ---
Total records: 1265657

--- Schema Check ---
root
 |-- trace_name: string (nullable = true)
 |-- network_code: string (nullable = true)
 |-- receiver_code: string (nullable = true)
 |-- receiver_type: string (nullable = true)
 |-- receiver_latitude: double (nullable = true)
 |-- receiver_longitude: double (nullable = true)
 |-- receiver_elevation_m: double (nullable = true)
 |-- p_arrival_sample: double (nullable = true)
 |-- p_status: string (nullable = true)
 |-- p_weight: double (nullable = true)
 |-- p_travel_sec: double (nullable = true)
 |-- s_arrival_sample: double (nullable = true)
 |-- s_status: string (nullable = true)
 |-- s_weight: double (nullable = true)
 |-- source_id: string (nullable = true)
 |-- source_origin_time: string (nullable = true)
 |-- source_origin_uncertainty_sec: double (nullable = true)
 |-- source_latitude: double (nullable = true)
 |-- source_longitude: dou

In [6]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
exec_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
exec_df['maxMemory_GB'] = (exec_df['maxMemory'] / (1024**3)).round(2)
exec_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,2388236697,0,True,2.22


## memeory allocation adventure
1. driver memory = 4, executor num = 4, executor mem = 31 : 37.271133422851562. driver memory = 4, executor num = 7, executor mem = 17 : 35.95031118392944
3. driver memory = 4, executor num = 6, executor mem = 20 : 33.44795203208923

